In [129]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import SparkSession, DataFrame
from pyspark.ml.feature import Imputer

In [ ]:
spark = (
    SparkSession.builder.appName("clean_dataset")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


In [131]:
IMPUTER_INPUT_COLS = ["anciennete_digitale_jours", "recence_gab_jours"]
IMPUTER_OUTPUT_COLS = ["anciennete_digitale_jours_imp", "recence_gab_jours_imp"]
IMPUTER_MODEL_PATH = "s3a://ml-scoring/models/imputer_anciennete_recence"


In [132]:
df = spark.read.parquet("part-00000-clean.parquet")
df.show(100, truncate=False)

+-------+------+------+-------+------+----+-------------+----------+---+--------+--------------+-------------+---------------+---------------+----------+-----------------+-----------+---------+-----------------------+-------------------+---------+----------+----------------------+------------------+------------------+------------------+-----------------+-----------------+-----------------+--------------------+-----------+----------------------+---------------------+--------------------+-------------------+----------------------+---------------------+-------------------------+------------------+-----------------+
|RADICAL|BANQUE|AGENCE|GENERIC|PLURAL|CCLE|DATE_OF_BIRTH|CODE_VILLE|BPR|GENDER  |MARITAL_STATUS|NOMBRE_ENFANT|CUSTOMER_RATING|TAILLE_ENTREPRI|label_code|label_nom        |pack_actuel|pack_etat|digital_toujours_abonne|solde_moyen        |solde_min|solde_max |nb_mois_observes_solde|depot_moyen       |flux_cred_moyen   |flux_cred_total   |nb_mois_avec_flux|nb_operations_gab|montan

# clean Data

In [133]:
df = df.withColumn(
    "GENDER",
    F.when(F.col("GENDER") == "FÃ©minin", "Féminin")
     .when(F.col("GENDER") == "Masculin", "Masculin")
     .otherwise(None)
)

In [134]:
for i, col in enumerate(df.columns):
    print(f"col {i}: {col} =>", df.filter(F.col(col).isNull()).count())


col 0: RADICAL => 0


col 1: BANQUE => 0
col 2: AGENCE => 0
col 3: GENERIC => 0
col 4: PLURAL => 0
col 5: CCLE => 0
col 6: DATE_OF_BIRTH => 0
col 7: CODE_VILLE => 0
col 8: BPR => 0
col 9: GENDER => 0
col 10: MARITAL_STATUS => 0
col 11: NOMBRE_ENFANT => 0
col 12: CUSTOMER_RATING => 0
col 13: TAILLE_ENTREPRI => 0
col 14: label_code => 0
col 15: label_nom => 0
col 16: pack_actuel => 986
col 17: pack_etat => 986
col 18: digital_toujours_abonne => 0
col 19: solde_moyen => 0
col 20: solde_min => 0
col 21: solde_max => 0
col 22: nb_mois_observes_solde => 0
col 23: depot_moyen => 0
col 24: flux_cred_moyen => 0
col 25: flux_cred_total => 0
col 26: nb_mois_avec_flux => 0
col 27: nb_operations_gab => 0
col 28: montant_total_gab => 0
col 29: montant_moyen_gab => 0
col 30: nb_retraits => 0
col 31: montant_total_retraits => 0
col 32: nb_paiements_digitaux => 0
col 33: montant_total_payfac => 0
col 34: nb_vignettes_payees => 0
col 35: montant_total_vignette => 0
col 36: jamais_active_digital => 0
col 37: anciennete_digita

In [135]:
df.select("anciennete_digitale_jours").distinct().show(truncate=False)

+-------------------------+
|anciennete_digitale_jours|
+-------------------------+
|4101                     |
|3997                     |
|5803                     |
|2366                     |
|2142                     |
|5300                     |
|3794                     |
|3918                     |
|1238                     |
|7754                     |
|7253                     |
|471                      |
|4929                     |
|1721                     |
|1084                     |
|623                      |
|6623                     |
|2811                     |
|1903                     |
|1143                     |
+-------------------------+
only showing top 20 rows


In [136]:
df.groupBy("anciennete_digitale_jours").count().show()

+-------------------------+-----+
|anciennete_digitale_jours|count|
+-------------------------+-----+
|                     4101|    2|
|                     3997|    1|
|                     5803|    2|
|                     2366|    1|
|                     2142|    2|
|                     5300|    1|
|                     3794|    1|
|                     3918|    1|
|                     1238|    1|
|                     7754|    1|
|                     7253|    1|
|                      471|    1|
|                     4929|    2|
|                     1721|    1|
|                     1084|    1|
|                      623|    1|
|                     6623|    1|
|                     2811|    1|
|                     1903|    1|
|                     1143|    1|
+-------------------------+-----+
only showing top 20 rows


In [137]:
df.filter(
    (F.col("jamais_active_digital") == 1) &
    (F.col("anciennete_digitale_jours").isNull())
).count()

1384

In [138]:
def clean_dataset(df: DataFrame) -> DataFrame:
    """
    Applique toutes les règles de nettoyage décidées pour ce projet.
    Chaque étape est commentée avec la décision métier qui la justifie
    (voir section 6.5 / 6.5bis du guide maître pour le détail).
    """

    n_avant = df.count()

    # --- 1. LIBELLE_VILLE : redondant avec CODE_VILLE (le code est fiable,
    #        0 null) -> on garde le code, on jette le libellé plutôt que
    #        d'imputer un texte qui n'apporte rien de plus au modèle ---
    if "LIBELLE_VILLE" in df.columns:
        df = df.drop("LIBELLE_VILLE")

    # --- 2. BPR / GENDER : nulls négligeables (2 et 1 lignes sur l'échantillon
    #        de référence) -> on supprime juste ces lignes, pas d'imputation ---
    subset_dropna = [c for c in ["BPR", "GENDER"] if c in df.columns]
    if subset_dropna:
        df = df.dropna(subset=subset_dropna)

    # --- 3. NOMBRE_ENFANT : null = pas d'enfant, PAS une valeur manquante
    #        à imputer par une médiane -> fillna(0) directement ---
    if "NOMBRE_ENFANT" in df.columns:
        df = df.fillna({"NOMBRE_ENFANT": 0})

    # --- 4. TAILLE_ENTREPRI : null = compte particulier (pas d'entreprise),
    #        pas une donnée manquante -> valeur catégorielle explicite.
    #        Devient un signal utile (particulier vs professionnel) au lieu
    #        d'être dropé ou imputé avec du bruit ---
    if "TAILLE_ENTREPRI" in df.columns:
        df = df.fillna({"TAILLE_ENTREPRI": "PARTICULIER"})

    # --- 4bis. pack_actuel / pack_etat : nulls groupés sur les mêmes clients
    #           (986 chacun) = clients sans pack digital, pas une vraie
    #           valeur manquante -> catégorie explicite, pas le mode ---
    pack_cols = {}
    if "pack_actuel" in df.columns:
        pack_cols["pack_actuel"] = "SANS_PACK"
    if "pack_etat" in df.columns:
        pack_cols["pack_etat"] = "SANS_ETAT"
    if pack_cols:
        df = df.fillna(pack_cols)

    # --- 5. depot_moyen / montant_moyen_gab : null = absence d'activité
    #        observée -> 0, cohérent avec le traitement déjà appliqué à
    #        flux_cred_moyen (NaN = 0, jamais une moyenne) ---
    montants_zero = [c for c in ["depot_moyen", "montant_moyen_gab"] if c in df.columns]
    if montants_zero:
        df = df.fillna({c: 0.0 for c in montants_zero})

    # --- 6. digital_date_activation : une date ne peut pas être mise à 0
    #        (serait confondu avec "activé aujourd'hui"). On dérive une
    #        ancienneté en jours + un flag explicite, puis on jette la
    #        date brute qui n'est de toute façon pas exploitable telle
    #        quelle par VectorAssembler ---
    if "digital_date_activation" in df.columns:
        df = (
            df.withColumn(
                "jamais_active_digital",
                F.when(F.col("digital_date_activation").isNull(), 1).otherwise(0),
            )
            .withColumn(
                "anciennete_digitale_jours",
                F.when(F.col("digital_date_activation").isNull(), F.lit(None))
                .otherwise(
                    F.datediff(
                        F.current_date(),
                        F.to_date("digital_date_activation", "dd/MM/yyyy"),
                    )
                ),
            )
            .drop("digital_date_activation")
        )

    # --- 7. derniere_operation_gab : même logique que #6, format datetime ---
    if "derniere_operation_gab" in df.columns:
        df = (
            df.withColumn(
                "jamais_utilise_gab",
                F.when(F.col("derniere_operation_gab").isNull(), 1).otherwise(0),
            )
            .withColumn(
                "recence_gab_jours",
                F.when(F.col("derniere_operation_gab").isNull(), F.lit(None))
                .otherwise(
                    F.datediff(
                        F.current_date(),
                        F.to_date(
                            F.col("derniere_operation_gab"), "dd/MM/yyyy HH:mm:ss"
                        ),
                    )
                ),
            )
            .drop("derniere_operation_gab")
        )

    n_apres = df.count()
    print(f"    Lignes avant : {n_avant} | après (dropna BPR/GENDER) : {n_apres}")

    return df

In [139]:
def run(path_in: str, path_out: str, label: str):
    print(f"\n{'=' * 20} NETTOYAGE : {label} {'=' * 20}")
    print(f"Lecture : {path_in}")
    df = spark.read.parquet(path_in)

    print("Nulls avant nettoyage :")
    df.select(
        [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]
    ).show(truncate=False, vertical=True)

    df_clean = clean_dataset(df)

    print("Nulls après nettoyage :")
    df_clean.select(
        [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_clean.columns]
    ).show(truncate=False, vertical=True)

    print(f"Écriture : {path_out}")
    df_clean.write.mode("overwrite").parquet(path_out)
    print(f"OK : {label} nettoyé et écrit.\n")


In [140]:
def apply_base_cleaning(path_in: str, label: str) -> DataFrame:
    print(f"\n{'=' * 20} NETTOYAGE DE BASE : {label} {'=' * 20}")
    print(f"Lecture : {path_in}")
    df = spark.read.parquet(path_in)

    print("Nulls avant nettoyage :")
    df.select(
        [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]
    ).show(truncate=False, vertical=True)

    df_clean = clean_dataset(df)
    return df_clean

In [141]:
def fit_and_apply_imputer_on_train(df_train: DataFrame) -> DataFrame:
    """
    Option A : Imputer (médiane) sur anciennete_digitale_jours / recence_gab_jours.
    Fit UNIQUEMENT sur le train (clients_avec_label) -- jamais sur la population
    à scorer, pour éviter toute fuite d'information et rester cohérent avec la
    logique déjà utilisée pour le Pipeline MLlib (fit sur train, transform partout).
    Le modèle d'imputation est sauvegardé pour être réappliqué tel quel sur
    dataset_a_scorer (mêmes médianes, pas recalculées).
    """
    cols_present = [c for c in IMPUTER_INPUT_COLS if c in df_train.columns]
    if not cols_present:
        return df_train

    out_cols = [IMPUTER_OUTPUT_COLS[IMPUTER_INPUT_COLS.index(c)] for c in cols_present]

    imputer = Imputer(inputCols=cols_present, outputCols=out_cols, strategy="median")
    imputer_model = imputer.fit(df_train)

    medianes = {c: df_train.approxQuantile(c, [0.5], 0.01)[0] for c in cols_present}
    print(f"Médianes apprises sur le train : {medianes}")

    df_train_imp = imputer_model.transform(df_train)

    print(f"Sauvegarde du modèle d'imputation : {IMPUTER_MODEL_PATH}")
    imputer_model.write().overwrite().save(IMPUTER_MODEL_PATH)

    return df_train_imp

In [142]:
def apply_saved_imputer(df_scorer: DataFrame) -> DataFrame:
    """Recharge l'Imputer entraîné sur le train et l'applique tel quel au
    dataset à scorer -- mêmes médianes des deux côtés, aucune fuite."""
    from pyspark.ml.feature import ImputerModel

    cols_present = [c for c in IMPUTER_INPUT_COLS if c in df_scorer.columns]
    if not cols_present:
        return df_scorer

    print(f"Chargement du modèle d'imputation : {IMPUTER_MODEL_PATH}")
    imputer_model = ImputerModel.load(IMPUTER_MODEL_PATH)
    return imputer_model.transform(df_scorer)

In [143]:
def show_nulls_and_write(df: DataFrame, path_out: str, label: str):
    print("Nulls après nettoyage complet (base + imputation) :")
    df.select(
        [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]
    ).show(truncate=False, vertical=True)

    print(f"Écriture : {path_out}")
    df.write.mode("overwrite").parquet(path_out)
    print(f"OK : {label} nettoyé et écrit.\n")

In [144]:
if __name__ == "__main__":
    # Adapter ces deux chemins si vos noms de dossiers Parquet diffèrent
    PATH_TRAIN_IN = "part-00000.parquet"
    PATH_TRAIN_OUT = "train_clean"

    # 1. Nettoyage de base + fit de l'Imputer SUR LE TRAIN UNIQUEMENT
    df_train = apply_base_cleaning(PATH_TRAIN_IN, "dataset_train_produits (clients avec label)")
    df_train = fit_and_apply_imputer_on_train(df_train)
    show_nulls_and_write(df_train, PATH_TRAIN_OUT, "dataset_train_produits (clients avec label)")

    print("\nTerminé. anciennete_digitale_jours_imp / recence_gab_jours_imp sont")
    print("désormais sans null (médianes apprises sur le train, réutilisées pour")
    print("le scoring). Utiliser les colonnes *_imp (pas les brutes) dans le")
    print("VectorAssembler, aux côtés des flags jamais_active_digital / jamais_utilise_gab.")


==================== NETTOYAGE DE BASE : dataset_train_produits (clients avec label) ====================
Lecture : part-00000.parquet
Nulls avant nettoyage :
-RECORD 0-----------------------
 RADICAL                 | 0    
 BANQUE                  | 0    
 AGENCE                  | 0    
 GENERIC                 | 0    
 PLURAL                  | 0    
 CCLE                    | 0    
 DATE_OF_BIRTH           | 0    
 CODE_VILLE              | 0    
 LIBELLE_VILLE           | 155  
 BPR                     | 2    
 GENDER                  | 1    
 MARITAL_STATUS          | 0    
 NOMBRE_ENFANT           | 267  
 CUSTOMER_RATING         | 0    
 TAILLE_ENTREPRI         | 4140 
 label_code              | 0    
 label_nom               | 0    
 pack_actuel             | 987  
 pack_etat               | 987  
 digital_date_activation | 1385 
 digital_toujours_abonne | 0    
 solde_moyen             | 0    
 solde_min               | 0    
 solde_max               | 0    
 nb_mois_observ

Py4JJavaError: An error occurred while calling o8961.save.
: java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2737)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3569)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3612)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:172)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3716)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3667)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:557)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:366)
	at org.apache.spark.ml.util.FileSystemOverwrite.handleOverwrite(ReadWrite.scala:833)
	at org.apache.spark.ml.util.MLWriter.save(ReadWrite.scala:175)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.ref